# Route I evaluation: direct inversion + flat velocity

This notebook runs the Route I baseline described in the paper draft:
- direct inversion from note-wise loudness parameters to MIDI velocity
- flat-velocity baseline
- shared audio-domain evaluation against both real audio and SoundFont(gt note, gt velocity)

The same evaluation backend is reusable for Route II / III / IV.


In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd
from IPython.display import display

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "score_hpt":
    REPO_ROOT = REPO_ROOT.parent
elif not (REPO_ROOT / "score_hpt").exists():
    raise RuntimeError("Please open this notebook from the repository root or from repo_root/score_hpt.")

SCORE_HPT_DIR = REPO_ROOT / "score_hpt"
PYTORCH_DIR = SCORE_HPT_DIR / "pytorch"
DATA_ANALYSIS_SRC = REPO_ROOT / "data_analysis" / "src"
for path in [PYTORCH_DIR, DATA_ANALYSIS_SRC]:
    path_str = str(path)
    if path_str not in sys.path:
        sys.path.insert(0, path_str)

print("REPO_ROOT:", REPO_ROOT)
print("PYTORCH_DIR:", PYTORCH_DIR)
print("DATA_ANALYSIS_SRC:", DATA_ANALYSIS_SRC)


## Parameters

Replace the dataset path and the SoundFont path before running.
- `STATS_JSON` should point to the dataset MIDI-statistics JSON used for percentile-to-velocity mapping.
- `INSTRUMENT_PATH` must point to a real `.sf2` or `.sfz` file.


In [ ]:
DATASET_TYPE = "smd"                  # "smd" | "maestro" | "maps"
DATASET_DIR = REPO_ROOT / "PATH_TO_DATASET"
STATS_JSON = REPO_ROOT / "data_analysis" / "stats" / "midi_sampler" / "SMD_sampler.json"
INSTRUMENT_PATH = REPO_ROOT / "PATH_TO_SOUNDFONT.sf2"   # replace with a real .sf2/.sfz file
OUT_DIR = SCORE_HPT_DIR / "outputs" / "route1_eval" / DATASET_TYPE

SPLIT = "test"
MAPS_PIANOS = "both"
FLAT_VELOCITY = 64
OVERWRITE = False

MAPPING_MODE = "rank_blend"          # "rank_blend" | "rank_harmonic" | "rank_flux" | "robust_blend"
HARMONIC_WEIGHT = 1.0
FLUX_WEIGHT = 0.25
FALLBACK_VELOCITY = 64

NOTE_SAMPLE_RATE = 22050
NOTE_FFT_SIZE = 2048
NOTE_HOP = 256
HARMONIC_COUNT = 5
BAND_BINS = 1
ENERGY_WINDOW_S = 0.12
ONSET_PRE_S = 0.02
ONSET_POST_S = 0.08

RENDER_SR = 44100
EVAL_SR = 22050
FPS = 50.0
FFT_SIZE = 1024
BSSL_MODE = "sone"
NUM_SAMPLES = 2048
NORMALIZATION = "zscore"
BACKEND = "auto"                     # auto -> .sf2 uses fluidsynth, .sfz uses sfizz
DEVICE = None                         # None | "cpu" | "cuda"


In [ ]:
from direct_invension.route1 import predict_route1_dataset
from direct_invension.eval_framework import evaluate_prediction_manifest, evaluation_results_to_dataframe

route1_run = predict_route1_dataset(
    dataset_type=DATASET_TYPE,
    dataset_dir=DATASET_DIR,
    out_dir=OUT_DIR / "predictions",
    dataset_stats_json=STATS_JSON,
    split=SPLIT,
    maps_pianos=MAPS_PIANOS,
    flat_velocity=FLAT_VELOCITY,
    mapping_mode=MAPPING_MODE,
    harmonic_weight=HARMONIC_WEIGHT,
    flux_weight=FLUX_WEIGHT,
    fallback_velocity=FALLBACK_VELOCITY,
    note_sample_rate=NOTE_SAMPLE_RATE,
    note_fft_size=NOTE_FFT_SIZE,
    note_hop=NOTE_HOP,
    harmonic_count=HARMONIC_COUNT,
    band_bins=BAND_BINS,
    energy_window_s=ENERGY_WINDOW_S,
    onset_pre_s=ONSET_PRE_S,
    onset_post_s=ONSET_POST_S,
    skip_existing=not OVERWRITE,
)
route1_run


In [ ]:
direct_eval = evaluate_prediction_manifest(
    route1_run["direct_manifest_path"],
    instrument_path=INSTRUMENT_PATH,
    out_dir=OUT_DIR / "evaluation_direct",
    render_sr=RENDER_SR,
    eval_sr=EVAL_SR,
    frames_per_second=FPS,
    fft_size=FFT_SIZE,
    bssl_mode=BSSL_MODE,
    num_samples=NUM_SAMPLES,
    normalization=NORMALIZATION,
    device=DEVICE,
    backend=BACKEND,
    skip_existing_render=not OVERWRITE,
)
direct_eval["summary"]


In [ ]:
flat_eval = evaluate_prediction_manifest(
    route1_run["flat_manifest_path"],
    instrument_path=INSTRUMENT_PATH,
    out_dir=OUT_DIR / f"evaluation_flat{FLAT_VELOCITY}",
    render_sr=RENDER_SR,
    eval_sr=EVAL_SR,
    frames_per_second=FPS,
    fft_size=FFT_SIZE,
    bssl_mode=BSSL_MODE,
    num_samples=NUM_SAMPLES,
    normalization=NORMALIZATION,
    device=DEVICE,
    backend=BACKEND,
    skip_existing_render=not OVERWRITE,
)
flat_eval["summary"]


In [ ]:
summary_df = evaluation_results_to_dataframe(direct_eval, flat_eval)
display(summary_df)
summary_df


## Notes

- Route I reports direct velocity MAE whenever GT note velocities exist in the MIDI.
- `real_ref_*` compares `SoundFont(gt note, pred velocity)` against real dataset audio.
- `synth_ref_*` compares `SoundFont(gt note, pred velocity)` against `SoundFont(gt note, gt velocity)`.
- The same `evaluate_prediction_manifest(...)` call is what Route II / III / IV should reuse.
